# 03 — Benchmarks: BERT vs DistilBERT

**Branch:** `reproduction/benchmarks`  
**Owner:** Person B or C

Loads saved results from notebooks 01 and 02 and produces the comparison table and plots
used in the presentation.

Run **after** notebooks 01 and 02 have saved their results to `../results/`.

In [ ]:
!pip install -q matplotlib seaborn pandas

In [ ]:
import sys
sys.path.append('../src')

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from utils import load_results

sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
# ── Load saved results ─────────────────────────────────────────────────────
bert      = load_results('bert_baseline.json')
distilbert = load_results('distilbert.json')

# Paper reference values
paper = {
    'bert':       {'val_accuracy': 93.5, 'total_parameters': 110_000_000, 'model_size_mb': 440},
    'distilbert': {'val_accuracy': 91.3, 'total_parameters': 66_000_000,  'model_size_mb': 265},
}

In [ ]:
# ── Comparison table ───────────────────────────────────────────────────────
rows = [
    {
        'Model':              'BERT-base (paper)',
        'Accuracy (%)':       paper['bert']['val_accuracy'],
        'Parameters':         f"{paper['bert']['total_parameters']:,}",
        'Size (MB)':          paper['bert']['model_size_mb'],
        'Speed (samples/s)':  'reference',
    },
    {
        'Model':              'DistilBERT (paper)',
        'Accuracy (%)':       paper['distilbert']['val_accuracy'],
        'Parameters':         f"{paper['distilbert']['total_parameters']:,}",
        'Size (MB)':          paper['distilbert']['model_size_mb'],
        'Speed (samples/s)':  '1.6x faster',
    },
    {
        'Model':              'BERT-base (ours)',
        'Accuracy (%)':       bert['val_accuracy'],
        'Parameters':         f"{bert['total_parameters']:,}",
        'Size (MB)':          bert['model_size_mb'],
        'Speed (samples/s)':  bert['inference_speed']['samples_per_second'],
    },
    {
        'Model':              'DistilBERT (ours)',
        'Accuracy (%)':       distilbert['val_accuracy'],
        'Parameters':         f"{distilbert['total_parameters']:,}",
        'Size (MB)':          distilbert['model_size_mb'],
        'Speed (samples/s)':  distilbert['inference_speed']['samples_per_second'],
    },
]

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── Plot 1: Accuracy comparison ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

models   = ['BERT-base\n(paper)', 'DistilBERT\n(paper)', 'BERT-base\n(ours)', 'DistilBERT\n(ours)']
accs     = [paper['bert']['val_accuracy'], paper['distilbert']['val_accuracy'],
            bert['val_accuracy'], distilbert['val_accuracy']]
colors   = ['#4C72B0', '#DD8452', '#4C72B0', '#DD8452']
hatches  = ['', '', '//', '//']

bars = ax.bar(models, accs, color=colors, hatch=hatches, edgecolor='white', width=0.5)
ax.set_ylim(88, 95)
ax.set_ylabel('SST-2 Accuracy (%)')
ax.set_title('BERT vs DistilBERT — SST-2 Accuracy')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{acc}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../results/accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Size vs Accuracy tradeoff ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

points = [
    ('BERT-base (ours)',    bert['total_parameters'] / 1e6,      bert['val_accuracy']),
    ('DistilBERT (ours)',   distilbert['total_parameters'] / 1e6, distilbert['val_accuracy']),
]

for name, params, acc in points:
    ax.scatter(params, acc, s=200, zorder=5)
    ax.annotate(name, (params, acc), textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('SST-2 Accuracy (%)')
ax.set_title('Model Size vs Accuracy Tradeoff')
plt.tight_layout()
plt.savefig('../results/tradeoff_plot.png', dpi=150)
plt.show()

In [ ]:
# ── Save combined benchmarks ───────────────────────────────────────────────
from utils import save_results

retention = round(distilbert['val_accuracy'] / bert['val_accuracy'] * 100, 1)
speedup   = round(distilbert['inference_speed']['samples_per_second'] /
                  bert['inference_speed']['samples_per_second'], 2)

benchmarks = {
    'bert_accuracy':        bert['val_accuracy'],
    'distilbert_accuracy':  distilbert['val_accuracy'],
    'accuracy_retention':   f'{retention}%',
    'speedup_factor':       f'{speedup}x',
    'bert_params':          bert['total_parameters'],
    'distilbert_params':    distilbert['total_parameters'],
    'param_reduction':      f"{round((1 - distilbert['total_parameters'] / bert['total_parameters']) * 100, 1)}%",
}

save_results(benchmarks, 'benchmarks.json')
print(benchmarks)